# Download of Sentinel-2 images from a list of sites

This notebook downloads the images of a list of events, one row per site, tile and period, e.g. a list of
field campaigns or of reported blooms. Here the list `Noctiluca_liste_grs.txt` of Noctiluca blooms on the
European coasts: the columns used are `process (yes if 1)`, `Site Name`, `start_date`, `end_date` and
`tile`.

The search and the download are those of `grstbx.download`, see the notebook *Download of Sentinel-2
images*: the search needs no account, the download needs the S3 keys of a CDSE account (`~/.s3cfg`).

In [1]:
import os

import pandas as pd

from grstbx import download

odir = '/data/satellite/Sentinel-2/L1C'     # root of the Level-1C images
max_cloud_cover = 60                         # %

## 1. List of sites

In [2]:
sites = pd.read_csv('Noctiluca_liste_grs.txt')
sites.columns = sites.columns.str.strip()
sites = sites[sites['process (yes if 1)'] == 1]
sites[['Site Name', 'start_date', 'end_date', 'tile']]

,Site Name,start_date,end_date,tile
0,Noctiluca_Mesodinium,2020-04-10,2020-04-12,30TWT
1,Noctiluca_Mesodinium,2020-04-10,2020-04-12,30TWS
2,Noctiluca_Mesodinium,2020-04-10,2020-04-12,30TWR
3,Noctiluca_Mesodinium,2020-04-10,2020-04-12,30TXS
4,Noctiluca_Mesodinium,2020-04-10,2020-04-12,30TXR
5,Noctiluca_Lepidodinium,2022-07-01,2022-07-03,30TWS
6,Noctiluca_Lepidodinium,2022-07-01,2022-07-03,30TXS
7,Noctiluca_Lingulaulax,2024-06-23,2024-06-25,30TVT
8,Noctiluca_Lingulaulax,2024-06-23,2024-06-25,30TWT
9,Noctiluca_NorthSea,2022-06-13,2022-06-25,32VMH


## 2. Images of each row

One search per row; the site name is kept with the images. An image of a tile can be in several rows (two
sites on the same tile): it is downloaded once.

In [3]:
found = []
for row in sites.itertuples():
    images = download.search(row.tile, row.start_date, row.end_date, max_cloud_cover=max_cloud_cover)
    found.append(images.assign(site=row._2))
images = pd.concat(found, ignore_index=True)
images['on_disk'] = download.on_disk(images, odir)

summary = images.groupby('site').agg(tiles=('tile', lambda t: ', '.join(sorted(set(t)))), images=('name', 'size'),
                                     GB=('size', lambda s: round(s.sum() / 1e9, 1)), on_disk=('on_disk', 'sum'))
unique = images.drop_duplicates('name')
print(f'{len(unique)} images, {unique.on_disk.sum()} already on disk, '
      f'{unique[~unique.on_disk]["size"].sum() / 1e9:.1f} GB to download')
summary

78 images, 53 already on disk, 14.6 GB to download


,tiles,images,GB,on_disk
site,,,,
Noctiluca_Brazil,22JGP,1,0.4,0
Noctiluca_CanadaBC,"10UDU, 10UDV",8,5.6,0
Noctiluca_Lepidodinium,"30TWS, 30TXS",2,1.2,0
Noctiluca_Lingulaulax,"30TVT, 30TWT",2,1.4,0
Noctiluca_Mesodinium,"30TWR, 30TWS, 30TWT, 30TXR, 30TXS",5,2.7,1
Noctiluca_NorthSea,"31UDA, 31UEA, 31UEB, 31UET, 31UEU, 31UEV, 31UF...",60,31.1,52


## 3. Download

Set `run_download` to `True`: the images already on disk are skipped, and the download can be resumed by
running the cell again.

In [4]:
run_download = False

if run_download:
    result = download.download(images.drop_duplicates('name'), odir, workers=4)
    print(result.status.value_counts().to_dict())
    print(result[result.status == 'failed'])

Or with s3cmd, from a shell (`xargs -P 4 -I {} sh -c "{}" < noctiluca_download.sh`):

In [5]:
commands = download.s3cmd_commands(images.drop_duplicates('name'), odir)
print(*commands[:2], sep='\n')
# with open('noctiluca_download.sh', 'w') as f:
#     f.write('\n'.join(commands) + '\n')

s3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2020/04/11/S2B_MSIL1C_20200411T110619_N0500_R137_T30TWT_20230509T002033.SAFE /data/satellite/Sentinel-2/L1C/30TWT/2020/04/11/
s3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2020/04/11/S2B_MSIL1C_20200411T110619_N0500_R137_T30TWS_20230509T002033.SAFE /data/satellite/Sentinel-2/L1C/30TWS/2020/04/11/
